In [ ]:
## After Data preparation (BG control samples) and signal injections, the known sources should be removed
## We use 2RXS catalog for catalog matching
#  Query from catalog takes time so we first make known source dictionary of observation id and the sources.
#　→ We refer the dictionary when we process source removal
# example of 2015


from astroquery.heasarc import Heasarc
from astropy.coordinates import SkyCoord
import astropy.units as u
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from scipy.optimize import minimize
from scipy.stats import poisson
import numpy as np
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.stats import poisson
from scipy.integrate import quad
from tqdm import tqdm
from scipy.optimize import minimize
from scipy.integrate import dblquad
import glob
import os
from scipy.optimize import brentq
from pathlib import Path
import re
import warnings
import time, random
import warnings
from http.client import RemoteDisconnected
import requests
from scipy.integrate import IntegrationWarning
warnings.filterwarnings("ignore", category=IntegrationWarning)
RETRY_EXC = (requests.exceptions.ConnectionError,
             requests.exceptions.ReadTimeout,
             RemoteDisconnected)


def xrt_known_sources(ra0_deg, dec0_deg): # “Search LSXPS sources within a 20-arcmin-radius area centered on the RA and Dec of the BG/Signal simulation files
    # fits files of living sxps catalog
    tbl = fits.open('/home/yu/XRT/catalog_data/slim/lsxps_slim.fits')[1].data

    half = 20  # arcmin
    dx_arcmin = (tbl['RA'] - ra0_deg) * np.cos(np.deg2rad(dec0_deg)) * 60.0
    dy_arcmin = (tbl['Decl'] - dec0_deg) * 60.0
    mask = (np.abs(dx_arcmin) <= half) & (np.abs(dy_arcmin) <= half)
    tbl = tbl[mask]

    RA  = tbl['RA'].astype(float).tolist()
    DEC = tbl['Decl'].astype(float).tolist()
    return RA, DEC, tbl

SNAME_RE = re.compile(r'(?i)sw\d{11}')

def extract_sname(name: str) -> str:
    m = SNAME_RE.search(name)
    if not m:
        raise ValueError(f"sname not found in: {name}")
    return m.group(0).lower()

# process one (sname: sw00000000xpcw3po_cl.evt→sw00000000)
def _build_one_row(
    sname: str,
    cl_dir: str,
):

    try:
        cl_path = Path(cl_dir) / f"{sname}xpcw3po_cl.evt"
        if not cl_path.exists():
            return (sname, np.array([], float), np.array([], float), None)

    
        with fits.open(cl_path, memmap=False, lazy_load_hdus=False) as ch:
            hdr = ch["EVENTS"].header
            ra0 = float(hdr["RA_PNT"])
            dec0 = float(hdr["DEC_PNT"])

        # HEASARC query
        RA, DEC, tbl = xrt_known_sources(
            ra0, dec0
        )
        return (sname, np.asarray(RA, float), np.asarray(DEC, float), tbl)

    except Exception as e:

        return (sname, np.array([], float), np.array([], float), None)

# -------- 並列ビルド関数 --------
def build_struct_array_from_snames_mp(
    snames,
    cl_dir="/home/yu/XRT/from_sciserver/2015/eventfiles_2015_single&1st_obs",
    max_workers=30):
    """
    dtype: [('sname','U12'), ('ra','O'), ('dec','O'), ('tbl','O')]
    """
    uniq = sorted(set(snames))
    rows = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = [
            ex.submit(_build_one_row, s, cl_dir)
            for s in uniq
        ]
        for fut in tqdm(as_completed(futs), total=len(futs)):
            rows.append(fut.result())

    dtype = np.dtype([('sname','O'), ('ra','O'), ('dec','O'), ('tbl','O')])
    arr = np.array(rows, dtype=dtype)
    return arr

def lookup_known(arr, sname: str):
    sname = sname.lower()
    hit = arr[arr['sname'] == sname]
    if hit.size == 0:
        raise KeyError(f"sname not found: {sname}")
    return hit['ra'][0], hit['dec'][0], hit['tbl'][0]

# ===== 使い方 =====
if __name__ == "__main__":
    evt_dir = Path("/home/yu/XRT/from_sciserver/2015/eventfiles_2015_single&1st_obs")
    snames = [extract_sname(p.name) for p in evt_dir.glob("*.evt")]

    arr = build_struct_array_from_snames_mp(
        snames,
        cl_dir="/home/yu/XRT/from_sciserver/2015/eventfiles_2015_single&1st_obs",
        max_workers=30,  # HEASARCに優しく
    )

    # 保存（Astropy Table を含むので allow_pickle=True 必須）
    np.save("/home/yu/XRT/from_sciserver/2015/2rxs_remove_data.npy", arr, allow_pickle=True)


## By combining the dictinary from 2013 to 2015, we make '/home/yu/XRT/MultipleTiling/swiftfiles/new_2013_2025_2rxs_remove_data.npy' 
  

In [ ]:
# 2rxs removal
# Source detection (same as source detection process of TS calculation) → Check RA,Dec → If they are near enough to the known sources, remove them.  
import sys
from astroquery.heasarc import Heasarc
from astropy.coordinates import SkyCoord
import astropy.units as u
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from scipy.optimize import minimize
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.stats import poisson
from scipy.integrate import quad
# from tqdm import tqdm
from scipy.integrate import dblquad
import glob
import os
from scipy.optimize import brentq
from pathlib import Path
import re
import warnings
import time, random
from http.client import RemoteDisconnected
import requests
from scipy.integrate import IntegrationWarning

warnings.filterwarnings("ignore", category=IntegrationWarning)
RETRY_EXC = (requests.exceptions.ConnectionError,
             requests.exceptions.ReadTimeout,
             RemoteDisconnected)

################# signal simulation parameters (which Γ and ρ) ################## 
LZGamma=7  # can be 3, 5, 7, and 10
# rho_list = np.array([2e-6, 4e-6, 6e-6, 8e-6, 2e-7, 4e-7, 6e-7, 8e-7, 2e-8, 4e-8, 6e-8, 8e-8, 2e-9, 4e-9, 6e-9, 8e-9, 2e-10])
rho_list = np.array([1e-10, 3e-10, 6e-10, 1e-9, 3e-9, 6e-9, 1e-8, 3e-8, 6e-8, 1e-7, 3e-7, 6e-7, 1e-6, 3e-6])
# rho_list = np.array([1e-9, 3e-9, 6e-9, 1e-8, 3e-8, 6e-8, 1e-7, 3e-7, 6e-7, 1e-6, 3e-6])
#################################################################################
merged = np.load('/home/yu/XRT/MultipleTiling/swiftfiles/new_2013_2025_2rxs_remove_data.npy', allow_pickle=True) # The known source position dictionary: obsid → RA, Dec of known sources that the observation contains
base_dir = Path('/home/yu/XRT/MultipleTiling')


_worker = {}
def _init_worker(merged):
    global _worker
    _worker = {"merged": merged}

def lookup_known(arr, sname: str):
    sname = sname.lower()
    hit = arr[arr['sname'] == sname]
    if hit.size == 0:
        raise KeyError(f"sname not found: {sname}")
    return hit['ra'][0], hit['dec'][0]



def RADECtoPIX(RA, DEC, RX, RY, DELX, DELY, RPX, RPY):
    """
    Inverse of conv_pix2RADEC.
    RA, DEC: degrees
    RX, RY : radians (reference RA/DEC)
    DELX, DELY: degrees per pixel
    RPX, RPY: reference pixel
    Returns x, y (pixels)
    """
    ra  = np.radians(RA)
    dec = np.radians(DEC)
    ra0 = RX
    dec0 = RY
    # wrap ΔRA to [-π, π] for numerical stability
    dra = (ra - ra0 + np.pi) % (2*np.pi) - np.pi
    A = np.cos(dec) * np.cos(dra)
    B = np.cos(dec) * np.sin(dra)
    C = np.sin(dec)
    D = np.sin(dec0)*C + np.cos(dec0)*A  # denominator
    # avoid divide-by-zero near the projection horizon
    eps = 1e-15
    D = np.where(np.abs(D) < eps, np.nan, D)
    # gnomonic plane coords consistent with your forward transform
    xi  = B / D
    eta = (np.cos(dec0)*C - np.sin(dec0)*A) / D
    X = xi          # radians
    Y = eta        # note the sign to match phi = atan2(X, -Y)
    x = RPX + np.degrees(X) / DELX
    y = RPY + np.degrees(Y) / DELY
    return x, y

def conv_RADEC2local(ra, dec, ra_pnt, dec_pnt, DELX, DELY, RPX, RPY):
    ra_rad = np.radians(ra)
    dec_rad = np.radians(dec)
    ra_pnt_rad = np.radians(ra_pnt)
    dec_pnt_rad = np.radians(dec_pnt)
    ca = np.cos(ra_pnt_rad)
    sa = np.sin(ra_pnt_rad)
    cd = np.cos(dec_pnt_rad)
    sd = np.sin(dec_pnt_rad)
    x = np.cos(dec_rad)*np.cos(ra_rad)
    y = np.cos(dec_rad)*np.sin(ra_rad)
    z = np.sin(dec_rad)
    x_ =    -sa*x +    ca*y
    y_ = -ca*sd*x - sa*sd*y + cd*z
    z_ =  ca*cd*x + sa*cd*y + sd*z
    x = RPX + np.degrees(x_) / DELX
    y = RPY + np.degrees(y_) / DELY
    return x, y


psffile = fits.open('/home/yu/XRT/caldb/data/swift/xrt/cpf/psf/swxpsf20010101v006.fits')
PC_006 = psffile['PC_PSF_COEF'].data
parameters_6 = PC_006['COEF0']
p0 = parameters_6[0]
p1 = parameters_6[1]
p2 = parameters_6[2]
p3 = parameters_6[3]
def psf(r):
    psf = p0*np.exp(-(r**2/2/p1**2))+(1-p0)*(1+(r/p2)**2)**-p3
    return psf
def psf_xy(y, x):
    r=np.sqrt(x**2+y**2)
    return psf(r)

def psf_integral(r):
    psf = 2*np.pi*r*(p0*np.exp(-(r**2/2/p1**2))+(1-p0)*(1+(r/p2)**2)**-p3)
    return psf


#########source detection#############
# find excess spot using big window whose center is the reported known source RA, Dec
# set small square as the source area and calculate how many photons are expected to be from the source
# determine removal radius R: ' outside R, no 2 and more photons are expected from the source'

big = 25 
small = 10
total, _ = quad(psf_integral, 0, np.inf)
# rsmall, _ = quad(psf_integral, 0,small)
rsmall, _ = dblquad(
    psf_xy,
    -small, small,  # xの範囲
    lambda x: -small,
    lambda x: small
)
def process_evt(known, evt_path):
    p = Path(evt_path)

    eventfile=fits.open(evt_path)
    events = eventfile['EVENTS'].data
    x = events['X']
    y = events['Y']
    
    sname = re.search(r'(?i)sw\d{11}', p.name)
    sname=sname.group(0).lower()
    clevt =fits.open(f'/home/yu/XRT/MultipleTiling/swiftfiles/eventfiles_2013_to_2025/{sname}xpcw3po_cl.evt')

    RA_PNT = clevt['EVENTS'].header["RA_PNT"]
    DEC_PNT = clevt['EVENTS'].header["DEC_PNT"]
    TCRPX2 = clevt['EVENTS'].header["TCRPX2"]    
    TCDLT2 = clevt['EVENTS'].header["TCDLT2"]    
    TCRPX3 = clevt['EVENTS'].header["TCRPX3"]    
    TCDLT3 = clevt['EVENTS'].header["TCDLT3"]    
    # RA, DEC, _ = 
    RA, DEC = lookup_known(known, sname)
    RA = np.array(RA)
    DEC = np.array(DEC)
    if len(RA)==0:
        return np.array([]), np.array([]), np.array([])
    xc_u, yc_u = conv_RADEC2local(RA, DEC, RA_PNT, DEC_PNT, TCDLT2, TCDLT3, TCRPX2, TCRPX3)
    expotime = 200 #s
    thr = 1 #/omega
    new_known_x_list =[]
    new_known_y_list = []
    R_list=[]
    for x_c, y_c in zip(xc_u, yc_u):
        xmax=round(x_c) + big
        xmin=round(x_c) - big
        ymax=round(y_c) + big
        ymin=round(y_c) - big
        omega = (2*big+1)**2
        x_sel = x[(x<=xmax)&(xmin<=x)&(y<=ymax)&(ymin<=y)]
        y_sel = y[(x<=xmax)&(xmin<=x)&(y<=ymax)&(ymin<=y)]

        # 初期値
        theta0 = np.array([x_c, y_c, 0.5])
        # print(theta0)
        # 境界（位置は領域内、alphaは[0,1]）
        bounds = [(xmin, xmax),
                (ymin, ymax),
                (0.0, 1.0)]
        def findsource(params):
            x_src, y_src, alpha = params
            dx=x_sel-x_src
            dy=y_sel-y_src
            
            result_vals = []
            for dx, dy in zip(dx, dy):
                r = np.hypot(dx, dy)

                    # psf を点評価（矩形の中心で）
                val = psf(r)
                result_vals.append(val / total)  # 正規化    
            result_vals=np.array(result_vals)
            # print(result_vals.shape)
            # print('sum', sum(result_vals))
            term = alpha *result_vals + (1-alpha) * (1 / omega)
            return -np.sum(np.log(term))

        res = minimize(
            findsource, theta0, method="L-BFGS-B", bounds=bounds,
            options={"maxiter": 200}
        )
        params=res.x
        x_known = params[0]
        y_known = params[1]
        alpha_fit = params[2]
        success = res.success
    
        # print(x_known, y_known, alpha_fit)

        new_xmax = round(x_known+small)
        new_xmin = round(x_known-small)
        new_ymax = round(y_known+small)
        new_ymin = round(y_known-small)
        new_x_sel = x[(x<=new_xmax)&(new_xmin<=x)&(y<=new_ymax)&(new_ymin<=y)]
        new_y_sel = y[(x<=new_xmax)&(new_xmin<=x)&(y<=new_ymax)&(new_ymin<=y)]
        # r = small
        # # dx = x - x_known
        # # dy = y - y_known
        # # mask = (dx*dx + dy*dy) <= (r*r)    # 半径rの円内（境界含む）
    
        # new_x_sel = x[mask]
        # new_y_sel = y[mask]
        def countscalc(params):
            x_src, y_src, alpha = params
            dx=new_x_sel-x_src
            dy=new_y_sel-y_src
            
            result_vals = []
            for dx, dy in zip(dx, dy):
                # ±0.5 ピクセルの矩形領域で二重積分
            #     val, _ = dblquad(
            #         psf_xy,
            #         dx - 0.5, dx + 0.5,  # xの範囲
            #         lambda x: dy - 0.5,
            #         lambda x: dy + 0.5
            #     )
            #     result_vals.append(val / total)  # 正規化    
            # result_vals=np.array(result_vals)
                r = np.hypot(dx, dy)

                    # psf を点評価（矩形の中心で）
                val = psf(r)
                result_vals.append(val / total)  # 正規化    
            result_vals=np.array(result_vals)

            term = alpha *result_vals + (1-alpha) * (1 / omega)
            return -np.sum(np.log(term))
        new_bounds = [(new_xmin, new_xmax),
                (new_ymin, new_ymax),
                (0.0, 1.0)]
        res = minimize(
            countscalc, theta0, method="L-BFGS-B", bounds=new_bounds,
            options={"maxiter": 200}
        )      
        new_known_x=res.x[0]
        new_known_y=res.x[1]
        alpha = res.x[2]
        N = len(new_x_sel)
        # print('N, alpha', N, alpha)
        cr_total = N*alpha*(total/rsmall)
        # print('cr_total', cr_total)
        
        def search_R(r):
            omega_thr=10**2
            mu=cr_total*(psf(r)/total)*omega_thr
            # p_2more = 1-(poisson.pmf(0, mu)+ poisson.pmf(1, mu))
            p_2more =poisson.sf(1, mu)
            return (2*np.pi*r/omega_thr)*p_2more
        A, _ = quad(search_R, 0, np.inf)
        a, b = 0.0, 1.0
        if (quad(search_R, a, np.inf)[0]==0) or (quad(search_R, b, np.inf)[0]<1):
            R=5
        else:
            while quad(search_R, b, np.inf)[0]-1>=0:
                b *= 2.0
            
            f = lambda R: quad(search_R, R, np.inf)[0]-1
            R = brentq(f, a, b)
            if R < 5:
                R=5
            # print('R', R)
        R_list.append(R)
        new_known_x_list.append(new_known_x)
        new_known_y_list.append(new_known_y)
        # target = total*(thr/cr_total) #total cr * pdf(r) = thr
        # print('target', target)
        # a, b = 0.0, 1.0
        # if psf(a) <= target:
        #      return 0.0
        # while psf(b) > target:
        #     b *= 2.0
        # f = lambda r: psf(r)-target
        # print('R', brentq(f, a, b))
        # new_known_x_list.append(new_known_x)
        # new_known_y_list.append(new_known_y)
        # R_list.append(brentq(f, a, b))
    known_source_x = np.array(new_known_x_list)
    known_source_y = np.array(new_known_y_list)
    R_list = np.array(R_list)

    return  known_source_x, known_source_y, R_list

# processing function 
def process_one(icdir, i, rho):
    known = _worker["merged"]
    icdir = Path(icdir)

    # lumis = ['2e45', '3e45', '4e45', '6e45', '7e45', '8e45', '9e45']
    # lumis = ['1e47', '1e48']
    if rho < 2e-9:
        lumis = ['1e44', '5e44', '1e45', '5e45', '6e45', '7e45', '8e45', '9e45', '1e46', '2e46', '3e46', '4e46', '5e46', '1e47']
    else:
        lumis = ['1e44', '5e44', '1e45', '2e45', '3e45', '4e45', '5e45', '6e45', '7e45', '8e45', '9e45', '1e46', '1e47']
    # lumis = ['BG']
    job_name = f"{icdir.name}/combined_{i}"

    R_list_local = []
    total_kept = 0
    total_events = 0

    for lumi in lumis:  
        # lumipath = icdir / "combined" / f"combined_{i}" / "BG"
        lumipath = icdir / f"combined/combined_{i}/unified_gamma{LZGamma:.0f}_modify_rng/{lumi}_{rho:.1e}/"
        if not lumipath.is_dir():
            continue

        injected_list = list(lumipath.glob("*.evt"))  # ← list化して長さチェック
        if len(injected_list) == 0:
            continue

        out_path = lumipath / "2rxs_removed"
        out_path.mkdir(parents=True, exist_ok=True)


        for evt_path in injected_list:
            p = Path(evt_path)
            base = p.stem  # 拡張子なし
            try:
                # 読み込み（閉じてから書くため memmap=False）
                with fits.open(p) as hdul:
                    ev = hdul["EVENTS"].data
                    # 既知ソース中心と半径（複数想定）
                    x0, y0, R = process_evt(known, evt_path)
                    
                    x0 = np.atleast_1d(np.asarray(x0, dtype=float))
                    y0 = np.atleast_1d(np.asarray(y0, dtype=float))
                    R  = np.atleast_1d(np.asarray(R,  dtype=float))
                    # print(len(x0))
                    if len(x0) == 0:  # known source が無いとき
                        mask_keep = np.ones(len(ev), dtype=bool)
                       
                    else:
                        dx = ev['X'][:, None] - x0[None, :]
                        dy = ev['Y'][:, None] - y0[None, :]
                        dist2 = dx**2 + dy**2
                        mask_keep = (dist2 > (R[None,:]**2)).all(axis=1)
                    ev_new = ev[mask_keep]
                    events_hdu = hdul["EVENTS"].copy()
                    events_hdu.data = ev_new
                    new_data = fits.HDUList([hdul[0], events_hdu, hdul['GTI'], hdul['BADPIX'], hdul['BIASDIFF']])
                    new_data.writeto(out_path / f'{base}.evt', overwrite=True)
                    kept = int(mask_keep.sum())
                    total = int(len(ev))

                    total_kept  += kept
                    total_events += total
                    R_list_local.extend(R)
            except Exception as e:
              
                print(f"[ERR in {p.name}] {e}")


    return (job_name, True, R_list_local, total_kept, total_events)


# arrays = [np.load(f, allow_pickle=True) for f in files]

# 1次元 or axis=0方向に全部連結




ic_dirs = [p for p in base_dir.glob(f"IC*") if p.is_dir()]

# Multiprocessing changing rho
for rho in rho_list:

    max_workers = 30
    R_list_full = []
    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=_init_worker,
        initargs=(merged,)      # ここで一度だけ known を渡す
    ) as ex:
        futs = []

        for icdir in ic_dirs:
            for i in tqdm(range(1000)):
                futs.append(ex.submit(process_one, icdir, i, float(rho)))

        for fut in tqdm(as_completed(futs), total=len(futs)):
            name, ok, R, *rest = fut.result()
            R_list_full.extend(R)
            if ok:
                kept, total = rest
                # print(f"[OK] {name}: kept {kept}/{total}")
            else:
                err = rest[0]
                print(f"[ERR] {name}: {err}")

